In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pathlib
import matplotlib.pyplot as plt
import time
import shutil

# Ensure you have the necessary libraries installed
# !pip install tensorflow-model-optimization

# Set the path to your dataset
# Make sure to replace 'path/to/your/dataset' with the actual path
data_dir = pathlib.Path(r'RiceLeafsDisease')

# Define image size and batch size
img_height = 224
img_width = 224
batch_size = 32

# Create training and validation datasets
# The dataset you provided has pre-split 'train' and 'validation' folders.
# We will use these directly instead of using validation_split.

train_dir = pathlib.Path(r'RiceLeafsDisease\train')

val_dir = pathlib.Path(r'RiceLeafsDisease\validation')


train_ds = tf.keras.utils.image_dataset_from_directory(
  train_dir,
  image_size=(img_height, img_width),
  batch_size=batch_size)

val_ds = tf.keras.utils.image_dataset_from_directory(
  val_dir,
  image_size=(img_height, img_width),
  batch_size=batch_size)

# Get the class names and number of classes
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names}")

# Configure the dataset for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 2100 files belonging to 6 classes.
Found 528 files belonging to 6 classes.
Number of classes: 6
Class names: ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scald', 'narrow_brown_spot']


In [5]:
# Load MobileNetV2 pre-trained on ImageNet
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

# Create the classification model
model = keras.Sequential([
    # Rescale pixel values from [0, 255] to [0, 1]
    layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(num_classes, activation='softmax')
])

# Compile and train the model
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

# Train the model
epochs = 10
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

# Save the full precision model
model.save('rice_leaf_fp32.h5')

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/10


c:\Users\pande\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\preprocessing\tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


66/66 ━━━━━━━━━━━━━━━━━━━━ 78s 968ms/step - accuracy: 0.6495 - loss: 0.9271 - val_accuracy: 0.7898 - val_loss: 0.5938
Epoch 2/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 59s 891ms/step - accuracy: 0.8567 - loss: 0.4401 - val_accuracy: 0.8561 - val_loss: 0.4338
Epoch 3/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 59s 890ms/step - accuracy: 0.8900 - loss: 0.3341 - val_accuracy: 0.8826 - val_loss: 0.3550
Epoch 4/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 58s 881ms/step - accuracy: 0.9176 - loss: 0.2744 - val_accuracy: 0.9053 - val_loss: 0.3092
Epoch 5/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 58s 876ms/step - accuracy: 0.9338 - loss: 0.2356 - val_accuracy: 0.9186 - val_loss: 0.2675
Epoch 6/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 61s 934ms/step - accuracy: 0.9452 - loss: 0.2035 - val_accuracy: 0.9205 - val_loss: 0.2461
Epoch 7/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 60s 913ms/step - accuracy: 0.9486 - loss: 0.1868 - val_accuracy: 0.9261 - val_loss: 0.2265
Epoch 8/10
66/66 ━━━━━━━━━━━━━━━━━━━━ 65s 987ms/step - accuracy: 0.9543 - loss: 0.1664 - val_accuracy: 0.926

In [7]:
# Load the trained full-precision model
model_fp32 = keras.models.load_model('rice_leaf_fp32.h5')

# Create a representative dataset for calibration
def representative_dataset_gen():
    # Iterate through a portion of your training data for calibration
    # We take a small number of samples (e.g., 100) from the dataset.
    for x, _ in train_ds.take(100):
        # We only need the image data, not the labels, for quantization.
        yield [x]

# Create the TFLite converter
converter = tf.lite.TFLiteConverter.from_keras_model(model_fp32)

# Enable quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Provide the representative dataset
converter.representative_dataset = representative_dataset_gen

# Ensure the inputs and outputs are in float
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.float32
converter.inference_output_type = tf.float32

# Convert the model
tflite_model_quant = converter.convert()

# Save the quantized TFLite model
with open('rice_leaf_quantized.tflite', 'wb') as f:
    f.write(tflite_model_quant)

INFO:tensorflow:Assets written to: C:\Users\pande\AppData\Local\Temp\tmp91eqwq7j\assets


INFO:tensorflow:Assets written to: C:\Users\pande\AppData\Local\Temp\tmp91eqwq7j\assets


Saved artifact at 'C:\Users\pande\AppData\Local\Temp\tmp91eqwq7j'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1242474972368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1242474971984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1242474972752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1242474972560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1242474971792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1242472249872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1242472247568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1242472250064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1242472248720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1242472247952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1242

c:\Users\pande\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [8]:
# Evaluate the full precision model
loss, fp32_accuracy = model_fp32.evaluate(val_ds)
print(f"Full Precision Model Accuracy: {fp32_accuracy * 100:.2f}%")

# Load the quantized TFLite model
interpreter = tf.lite.Interpreter(model_path="rice_leaf_quantized.tflite")
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Evaluate the TFLite model
def evaluate_tflite_model(interpreter, dataset):
    correct_predictions = 0
    total_predictions = 0
    
    # Pre-allocate input tensor to avoid overhead
    input_shape = input_details[0]['shape']
    
    for images, labels in dataset:
        for i in range(images.shape[0]):
            input_tensor = np.expand_dims(images[i].numpy(), axis=0).astype(np.float32)
            interpreter.set_tensor(input_details[0]['index'], input_tensor)
            interpreter.invoke()
            output_data = interpreter.get_tensor(output_details[0]['index'])
            predicted_class = np.argmax(output_data)
            
            if predicted_class == labels[i].numpy():
                correct_predictions += 1
            total_predictions += 1
            
    accuracy = correct_predictions / total_predictions
    return accuracy

quantized_accuracy = evaluate_tflite_model(interpreter, val_ds)
print(f"Quantized Model Accuracy: {quantized_accuracy * 100:.2f}%")

# Get file sizes
fp32_size = os.path.getsize('rice_leaf_fp32.h5') / 1024**2 # in MB
quantized_size = os.path.getsize('rice_leaf_quantized.tflite') / 1024**2 # in MB

print(f"\nFull Precision Model Size: {fp32_size:.2f} MB")
print(f"Quantized Model Size: {quantized_size:.2f} MB")
print(f"Size Reduction: {(1 - quantized_size / fp32_size) * 100:.2f}%")

17/17 ━━━━━━━━━━━━━━━━━━━━ 18s 794ms/step - accuracy: 0.9375 - loss: 0.1931
Full Precision Model Accuracy: 93.75%


c:\Users\pande\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Quantized Model Accuracy: 82.77%

Full Precision Model Size: 9.05 MB
Quantized Model Size: 2.59 MB
Size Reduction: 71.42%


In [10]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Get true labels and predictions from the validation dataset
true_labels = []
predictions = []

for images, labels in val_ds:
    # Get predictions for the entire batch
    batch_predictions = model_fp32.predict(images)
    
    # Append true labels
    true_labels.extend(labels.numpy())
    
    # Get the class with the highest probability
    predicted_classes = np.argmax(batch_predictions, axis=1)
    predictions.extend(predicted_classes)

# Convert lists to NumPy arrays
true_labels = np.array(true_labels)
predictions = np.array(predictions)

# Generate and print the classification report
print("Full-Precision Model Classification Report:")
print(classification_report(true_labels, predictions, target_names=class_names))

# Generate and print the confusion matrix
cm = confusion_matrix(true_labels, predictions)
print("\nFull-Precision Model Confusion Matrix:")
print(cm)

# You can also manually calculate F1, Precision, and Recall for specific classes if needed
from sklearn.metrics import f1_score, precision_score, recall_score

macro_precision = precision_score(true_labels, predictions, average='macro')
macro_recall = recall_score(true_labels, predictions, average='macro')
macro_f1 = f1_score(true_labels, predictions, average='macro')

print(f"\nMacro-averaged Precision: {macro_precision:.4f}")
print(f"Macro-averaged Recall: {macro_recall:.4f}")
print(f"Macro-averaged F1-Score: {macro_f1:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 937ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 889ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 779ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 758ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 770ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 787ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 719ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 749ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 716ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 712ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 720ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 731ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 750ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 711ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 739ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Full-Precision Model Classification Report:
                       precision    recall  f1-score   support

bacterial_leaf_blight       0.97      1.00      0.98        88
           brown_spot       0.89      0.82      0.85        88
              healthy       0.94      0.97      0.96        88
           leaf_blast       0.83      0.90    

In [11]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Load the TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_path="rice_leaf_quantized.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Get true labels and predictions
true_labels_tflite = []
predictions_tflite = []

for images, labels in val_ds:
    for i in range(images.shape[0]):
        # Prepare a single image for inference
        input_tensor = np.expand_dims(images[i].numpy(), axis=0).astype(np.float32)
        interpreter.set_tensor(input_details[0]['index'], input_tensor)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details[0]['index'])
        
        # Get the predicted class
        predicted_class = np.argmax(output_data)
        
        # Append true labels and predictions
        true_labels_tflite.append(labels[i].numpy())
        predictions_tflite.append(predicted_class)

# Convert lists to NumPy arrays
true_labels_tflite = np.array(true_labels_tflite)
predictions_tflite = np.array(predictions_tflite)

# Generate and print the classification report
print("\nQuantized TFLite Model Classification Report:")
print(classification_report(true_labels_tflite, predictions_tflite, target_names=class_names))

# Generate and print the confusion matrix
cm_tflite = confusion_matrix(true_labels_tflite, predictions_tflite)
print("\nQuantized TFLite Model Confusion Matrix:")
print(cm_tflite)

# You can also manually calculate F1, Precision, and Recall
macro_precision_tflite = precision_score(true_labels_tflite, predictions_tflite, average='macro')
macro_recall_tflite = recall_score(true_labels_tflite, predictions_tflite, average='macro')
macro_f1_tflite = f1_score(true_labels_tflite, predictions_tflite, average='macro')

print(f"\nMacro-averaged Precision: {macro_precision_tflite:.4f}")
print(f"Macro-averaged Recall: {macro_recall_tflite:.4f}")
print(f"Macro-averaged F1-Score: {macro_f1_tflite:.4f}")

c:\Users\pande\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



Quantized TFLite Model Classification Report:
                       precision    recall  f1-score   support

bacterial_leaf_blight       0.75      1.00      0.85        88
           brown_spot       0.89      0.67      0.77        88
              healthy       0.93      0.91      0.92        88
           leaf_blast       0.64      0.89      0.74        88
           leaf_scald       0.95      0.94      0.95        88
    narrow_brown_spot       1.00      0.56      0.72        88

             accuracy                           0.83       528
            macro avg       0.86      0.83      0.82       528
         weighted avg       0.86      0.83      0.82       528


Quantized TFLite Model Confusion Matrix:
[[88  0  0  0  0  0]
 [ 8 59  3 18  0  0]
 [ 1  0 80  7  0  0]
 [ 2  3  3 78  2  0]
 [ 5  0  0  0 83  0]
 [14  4  0 19  2 49]]

Macro-averaged Precision: 0.8606
Macro-averaged Recall: 0.8277
Macro-averaged F1-Score: 0.8245


Dynamic Range Quantization (DRQ) Model

In [12]:
import tensorflow as tf
from tensorflow import keras
import os

# Load the trained full-precision model
model_fp32 = keras.models.load_model('rice_leaf_fp32.h5')

# Create the TFLite converter
converter = tf.lite.TFLiteConverter.from_keras_model(model_fp32)

# Enable Dynamic Range Quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Convert the model
tflite_model_drq = converter.convert()

# Save the TFLite model with DRQ
with open('rice_leaf_drq.tflite', 'wb') as f:
    f.write(tflite_model_drq)

# You can now evaluate this model using the same function as before
# (evaluate_tflite_model).

INFO:tensorflow:Assets written to: C:\Users\pande\AppData\Local\Temp\tmptdpi708i\assets


INFO:tensorflow:Assets written to: C:\Users\pande\AppData\Local\Temp\tmptdpi708i\assets


Saved artifact at 'C:\Users\pande\AppData\Local\Temp\tmptdpi708i'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  1247287901648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1247287900688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1247287899152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1247287902032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1247287900880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1247287901456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1247287901264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1247287900112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1247287900496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1247287901840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1244

In [13]:
# Evaluate the full precision model
loss, fp32_accuracy = model_fp32.evaluate(val_ds)
print(f"Full Precision Model Accuracy: {fp32_accuracy * 100:.2f}%")

# Load the quantized TFLite model
interpreter = tf.lite.Interpreter(model_path="rice_leaf_drq.tflite")
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Evaluate the TFLite model
def evaluate_tflite_model(interpreter, dataset):
    correct_predictions = 0
    total_predictions = 0
    
    # Pre-allocate input tensor to avoid overhead
    input_shape = input_details[0]['shape']
    
    for images, labels in dataset:
        for i in range(images.shape[0]):
            input_tensor = np.expand_dims(images[i].numpy(), axis=0).astype(np.float32)
            interpreter.set_tensor(input_details[0]['index'], input_tensor)
            interpreter.invoke()
            output_data = interpreter.get_tensor(output_details[0]['index'])
            predicted_class = np.argmax(output_data)
            
            if predicted_class == labels[i].numpy():
                correct_predictions += 1
            total_predictions += 1
            
    accuracy = correct_predictions / total_predictions
    return accuracy

quantized_accuracy = evaluate_tflite_model(interpreter, val_ds)
print(f"Quantized Model Accuracy: {quantized_accuracy * 100:.2f}%")

# Get file sizes
fp32_size = os.path.getsize('rice_leaf_fp32.h5') / 1024**2 # in MB
quantized_size = os.path.getsize('rice_leaf_quantized.tflite') / 1024**2 # in MB

print(f"\nFull Precision Model Size: {fp32_size:.2f} MB")
print(f"Quantized Model Size: {quantized_size:.2f} MB")
print(f"Size Reduction: {(1 - quantized_size / fp32_size) * 100:.2f}%")

17/17 ━━━━━━━━━━━━━━━━━━━━ 17s 804ms/step - accuracy: 0.9375 - loss: 0.1931
Full Precision Model Accuracy: 93.75%


c:\Users\pande\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Quantized Model Accuracy: 92.23%

Full Precision Model Size: 9.05 MB
Quantized Model Size: 2.59 MB
Size Reduction: 71.42%


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Load the TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_path="rice_leaf_drq.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Get true labels and predictions
true_labels_tflite = []
predictions_tflite = []

for images, labels in val_ds:
    for i in range(images.shape[0]):
        # Prepare a single image for inference
        input_tensor = np.expand_dims(images[i].numpy(), axis=0).astype(np.float32)
        interpreter.set_tensor(input_details[0]['index'], input_tensor)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details[0]['index'])
        
        # Get the predicted class
        predicted_class = np.argmax(output_data)
        
        # Append true labels and predictions
        true_labels_tflite.append(labels[i].numpy())
        predictions_tflite.append(predicted_class)

# Convert lists to NumPy arrays
true_labels_tflite = np.array(true_labels_tflite)
predictions_tflite = np.array(predictions_tflite)

# Generate and print the classification report
print("\nQuantized TFLite Model Classification Report:")
print(classification_report(true_labels_tflite, predictions_tflite, target_names=class_names))

# Generate and print the confusion matrix
cm_tflite = confusion_matrix(true_labels_tflite, predictions_tflite)
print("\nDRQ Quantized TFLite Model Confusion Matrix:")
print(cm_tflite)

# You can also manually calculate F1, Precision, and Recall
macro_precision_tflite = precision_score(true_labels_tflite, predictions_tflite, average='macro')
macro_recall_tflite = recall_score(true_labels_tflite, predictions_tflite, average='macro')
macro_f1_tflite = f1_score(true_labels_tflite, predictions_tflite, average='macro')

print(f"\nMacro-averaged Precision: {macro_precision_tflite:.4f}")
print(f"Macro-averaged Recall: {macro_recall_tflite:.4f}")
print(f"Macro-averaged F1-Score: {macro_f1_tflite:.4f}")


Quantized TFLite Model Classification Report:
                       precision    recall  f1-score   support

bacterial_leaf_blight       0.97      1.00      0.98        88
           brown_spot       0.95      0.70      0.81        88
              healthy       0.92      0.95      0.94        88
           leaf_blast       0.76      0.91      0.83        88
           leaf_scald       0.98      1.00      0.99        88
    narrow_brown_spot       0.99      0.97      0.98        88

             accuracy                           0.92       528
            macro avg       0.93      0.92      0.92       528
         weighted avg       0.93      0.92      0.92       528


Quantized TFLite Model Confusion Matrix:
[[88  0  0  0  0  0]
 [ 3 62  3 19  0  1]
 [ 0  0 84  4  0  0]
 [ 0  3  4 80  1  0]
 [ 0  0  0  0 88  0]
 [ 0  0  0  2  1 85]]

Macro-averaged Precision: 0.9287
Macro-averaged Recall: 0.9223
Macro-averaged F1-Score: 0.9212
